In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0,
)

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader_a1 = PyPDFLoader("Basel_iii_ Past_Regulation_2019.pdf")
loader_a2 = PyPDFLoader("Basel_iii_Updated_Regulation_2026.pdf")
loader_a3 = PyPDFLoader("AxisBank_report_march-2025.pdf")

docs_a1 = loader_a1.load()
docs_a2 = loader_a2.load()
docs_a3 = loader_a3.load()

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunks_a1 = text_splitter.split_documents(docs_a1)
chunks_a2 = text_splitter.split_documents(docs_a2)
chunks_a3 = text_splitter.split_documents(docs_a3)

In [4]:
from langchain_core.prompts import PromptTemplate

prompt_extract_old = PromptTemplate.from_template(
    """
    ###SCRAPED TEXT FROM PDF:
    {docs}
    ###INSTRUCTION:
    The scraped text is from the Basel III SRP 98 documentation (2019 version).
    Your job is to extract each disclosure requirement with the following fields:
    - `requirement_title`: A short title of the disclosure requirement.
    - `description`: A concise summary of what the requirement entails.
    - `category`: One of: "Capital", "Liquidity", "Risk-Weighted-Assets", "Leverage", "Credit Risk", "Market Risk", "Operational Risk", or "Other".
    - `frequency`: How often the disclosure must be made (e.g., quarterly, annually).
    - `data_required`: A list of key data points needed to fulfill the requirement.
    
    ###IMPORTANT:
    - Do not add any explanation.
    - Do not include triple backticks or formatting.
    - Return only a raw, valid JSON array with multiple items.
    - No introduction, no commentary.

    ###VALID JSON FORMAT:
    [
      {{
        "requirement_title": "...",
        "description": "...",
        "category": "...",
        "frequency": "...",
        "data_required": ["...", "..."]
      }},
      ...
    ]
    """
)

prompt_extract_new = PromptTemplate.from_template(
    """
    ###SCRAPED TEXT FROM PDF:
    {docs}
    ###INSTRUCTION:
    The text is from the *updated* Basel III SRP 98 regulations (2026 version).
    Extract disclosure requirements with the following fields:
    - `requirement_title`
    - `description`
    - `category` (choose from given list)
    - `frequency`
    - `data_required`
    
    ###IMPORTANT:
    - Capture only the new or updated requirements relevant to Pillar 3.
    - Do not add explanations or any markdown formatting.
    - Return a valid JSON array only.

    ###CATEGORY OPTIONS:
    - "Capital", "Liquidity", "Risk-Weighted-Assets", "Leverage", "Credit Risk", "Market Risk", "Operational Risk", "Other"

    ###VALID JSON FORMAT:
    [
      {{
        "requirement_title": "...",
        "description": "...",
        "category": "...",
        "frequency": "...",
        "data_required": ["...", "..."]
      }}
    ]
    """
)

prompt_extract_axis_fixed = PromptTemplate.from_template(
    """
    ### RAW TEXT FROM PDF (Basel III Disclosure Report):
    {docs}

    ### TASK:
    Extract structured disclosure data related to Basel III Pillar 3 using the following format:

    [
      {{
        "section_title": "...",
        "metrics_provided": ["...", "..."],
        "content_summary": "...",
        "frequency_reported": "quarterly/annually/etc"
      }}
    ]

    ### RULES:
    - DO NOT add any explanation before or after.
    - DO NOT use markdown or wrap output in triple backticks.
    - DO NOT include any notes or commentary.
    - If no valid data, return: `[]`
    - Return ONLY a **valid raw JSON array** as shown above.
    """
)

In [5]:
import json
import time
import re
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from tqdm import tqdm

def process_chunks(chunks, prompt_template: PromptTemplate, output_path: str, delay=2, retries=3):
    chain = prompt_template | llm | StrOutputParser()
    extracted_data = []

    print(f"Processing {len(chunks)} chunks...")

    for i, chunk in enumerate(tqdm(chunks)):
        success = False

        for attempt in range(retries):
            try:
                result = chain.invoke({"docs": chunk.page_content}).strip()

                # 👇 Optional: Strip triple backticks or markdown wrappers
                if "```" in result:
                    parts = result.split("```")
                    for part in parts:
                        part = part.strip()
                        if part.startswith("[") and part.endswith("]"):
                            result = part
                            break  # Use this clean JSON part

                # 👇 Optional: Remove any prefix text like explanations
                if not result.startswith("["):
                    first_bracket = result.find("[")
                    last_bracket = result.rfind("]")
                    if first_bracket != -1 and last_bracket != -1:
                        result = result[first_bracket:last_bracket+1]

                # DEBUG print
                print(f"[Chunk {i} - Attempt {attempt+1}] Cleaned LLM Output:\n{result[:300]}")

                # ✅ Try parsing cleaned result

                # Use regex to extract the first JSON array only
                match = re.search(r'\[\s*{.*?}\s*\]', result, re.DOTALL)
                if match:
                    result = match.group(0)  # Only valid JSON array
                else:
                    raise ValueError("No valid JSON array found")
                json_obj = json.loads(result)
                
                if isinstance(json_obj, list):
                    extracted_data.extend(json_obj)
                else:
                    print(f"[Chunk {i}] ❗ Got non-list JSON: {type(json_obj)}")

                success = True
                break  # break out of retry loop

            except Exception as e:
                print(f"[Chunk {i} - Attempt {attempt+1}] ❌ Error: {e}")
                time.sleep(delay * (attempt + 1))  # exponential backoff

        if not success:
            print(f"[Chunk {i}] ❌ Skipped after {retries} attempts.")

        time.sleep(delay)  # throttle to avoid rate limit

    # Save to JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(extracted_data, f, indent=2)
    print(f"✅ Saved extracted data to {output_path}")


In [ ]:
import os 
os.makedirs("json", exist_ok=True)

print("Starting extraction...")

process_chunks(chunks_a1, prompt_extract_old, "json/basel_2019_extracted.json")
# process_chunks(chunks_a2, prompt_extract_new, "json/basel_2026_extracted.json")
# process_chunks(chunks_a3, prompt_extract_report, "json/axis_2025_extracted.json")

print("✅ All done")

In [ ]:
process_chunks(chunks_a2, prompt_extract_new, "json/basel_2026_extracted.json")

In [ ]:
process_chunks(chunks_a3, prompt_extract_report, "json/axis_2025_extracted.json")

In [6]:
import json
import pandas as pd

with open("json/basel_2026_extracted.json") as f:
    a2_data = json.load(f)

with open("json/axis_2025_extracted.json") as f:
    a3_data = json.load(f)

df_a2 = pd.DataFrame(a2_data)  # Updated regulation
df_a3 = pd.DataFrame(a3_data)  # Axis report


In [7]:
print("df_a2 columns:", df_a2.columns.tolist())
print("df_a3 columns:", df_a3.columns.tolist())

df_a2 columns: ['requirement_title', 'description', 'category', 'frequency', 'data_required']
df_a3 columns: ['section_title', 'metrics_provided', 'content_summary', 'frequency_reported']


In [8]:
df_a2['requirement_title_clean'] = df_a2['requirement_title'].str.lower().str.strip()
df_a3['section_title_clean'] = df_a3['section_title'].str.lower().str.strip()

In [9]:
# Find updated Basel requirements (a2) not covered in Axis report (a3)
missing = df_a2[~df_a2['requirement_title_clean'].isin(df_a3['section_title_clean'])]

# Show number of missing requirements
print(f"🔍 Found {len(missing)} disclosures missing in Axis Bank report")


🔍 Found 73 disclosures missing in Axis Bank report


In [12]:
missing.head(10)  # preview in notebook

,requirement_title,description,category,frequency,data_required,requirement_title_clean
0,Interest Rate Risk in the Banking Book,Application guidance on interest rate risk in ...,Market Risk,Annual,[Interest rate risk in the banking book],interest rate risk in the banking book
1,Disclosure of capital adequacy,Disclosure of the institution's capital adequacy,Capital,Annual,"[Capital adequacy ratio, Common equity tier 1 ...",disclosure of capital adequacy
2,Disclosure of risk-weighted assets,Disclosure of the institution's risk-weighted ...,Risk-Weighted-Assets,Annual,"[Risk-weighted assets, Risk-weighted assets by...",disclosure of risk-weighted assets
3,Disclosure of leverage ratio,Disclosure of the institution's leverage ratio,Leverage,Quarterly,"[Leverage ratio, Tier 1 capital, Total assets]",disclosure of leverage ratio
4,Disclosure of credit risk,Disclosure of the institution's credit risk,Credit Risk,Annual,"[Credit risk exposure, Credit risk weighted as...",disclosure of credit risk
5,Disclosure of market risk,Disclosure of the institution's market risk,Market Risk,Annual,"[Market risk exposure, Market risk weighted as...",disclosure of market risk
6,Disclosure of operational risk,Disclosure of the institution's operational risk,Operational Risk,Annual,"[Operational risk exposure, Operational risk w...",disclosure of operational risk
7,Definition of interest rate risk in the bankin...,Definition of interest rate risk in the bankin...,Market Risk,N/A,[],definition of interest rate risk in the bankin...
8,Valuation of banking book items,Two distinct methods for valuing banking book ...,Risk-Weighted-Assets,Not applicable,[],valuation of banking book items
9,Disclosure of accounting values of fair valued...,Accounting values of fair valued instruments c...,Risk-Weighted-Assets,Periodic,[Accounting values of fair valued instruments],disclosure of accounting values of fair valued...


In [13]:
missing['category'].value_counts()

category
Market Risk             25
Capital                 18
Risk-Weighted-Assets    13
Credit Risk             11
Liquidity                3
Leverage                 1
Operational Risk         1
Other                    1
Name: count, dtype: int64

In [15]:
market_risk_missing = missing[missing["category"] == "Market Risk"].reset_index(drop=True)
market_risk_missing.head(3)  # preview top 3


,requirement_title,description,category,frequency,data_required,requirement_title_clean
0,Interest Rate Risk in the Banking Book,Application guidance on interest rate risk in ...,Market Risk,Annual,[Interest rate risk in the banking book],interest rate risk in the banking book
1,Disclosure of market risk,Disclosure of the institution's market risk,Market Risk,Annual,"[Market risk exposure, Market risk weighted as...",disclosure of market risk
2,Definition of interest rate risk in the bankin...,Definition of interest rate risk in the bankin...,Market Risk,N/A,[],definition of interest rate risk in the bankin...


In [14]:
from langchain_core.prompts import PromptTemplate

market_risk_prompt = PromptTemplate.from_template(
    """
You are a Basel III compliance reporting assistant.
Based on the following disclosure requirement, generate a sample disclosure section for a bank's Pillar 3 report.

### REQUIREMENT:
- Title: {requirement_title}
- Description: {description}
- Frequency: {frequency}
- Data Required: {data_required}

### TASK:
Generate a realistic disclosure block with:
1. A disclosure **heading/title**
2. A **1-paragraph summary** explaining the requirement
3. A list of **placeholder metrics** in bullet points or table form
4. Make sure the output looks like it belongs in a bank’s Basel III Pillar 3 Market Risk section.

Return raw text only. No markdown.
"""
)


In [ ]:
from tqdm import tqdm

market_risk_drafts = []

for i, row in tqdm(market_risk_missing.iterrows(), total=len(market_risk_missing)):
    try:
        prompt = market_risk_prompt.format(
            requirement_title=row['requirement_title'],
            description=row['description'],
            frequency=row['frequency'],
            data_required=", ".join(row['data_required']) if isinstance(row['data_required'], list) else row['data_required']
        )

        result = llm.invoke(prompt)
        market_risk_drafts.append({
            "requirement_title": row['requirement_title'],
            "generated_disclosure": result.content.strip()
        })

    except Exception as e:
        print(f"[Error on row {i}] {e}")


In [ ]:
import os
import json

os.makedirs("output", exist_ok=True)

with open("output/market_risk_generated_disclosures.json", "w", encoding="utf-8") as f:
    json.dump(market_risk_drafts, f, indent=2)

print("✅ Saved Market Risk disclosures to output/market_risk_generated_disclosures.json")


In [3]:
import pandas as pd
from difflib import SequenceMatcher

# Load the extracted JSONs
with open("json/basel_2019_extracted.json") as f:
    a1 = pd.DataFrame(json.load(f))
with open("json/basel_2026_extracted.json") as f:
    a2 = pd.DataFrame(json.load(f))

# Normalize
a1["requirement_title_clean"] = a1["requirement_title"].str.lower().str.strip()
a2["requirement_title_clean"] = a2["requirement_title"].str.lower().str.strip()

# Find updated or new requirements
def is_similar(title, others, threshold=0.85):
    return any(SequenceMatcher(None, title, other).ratio() > threshold for other in others)

a1_titles = a1["requirement_title_clean"].tolist()
a2["is_new_or_updated"] = ~a2["requirement_title_clean"].apply(lambda t: is_similar(t, a1_titles))

# Filter only new/updated
a2_updates = a2[a2["is_new_or_updated"]].reset_index(drop=True)

# Save for next step
os.makedirs("json", exist_ok=True)
a2_updates.to_json("json/a2_differences.json", orient="records", indent=2)
print("✅ Saved new/updated disclosures to `json/a2_differences.json`")


✅ Saved new/updated disclosures to `json/a2_differences.json`


In [4]:
import json
import pandas as pd
from difflib import SequenceMatcher

# Load new/updated requirements and axis report
with open("json/a2_differences.json") as f:
    a2_updates = pd.DataFrame(json.load(f))
with open("json/axis_2025_extracted.json") as f:
    axis = pd.DataFrame(json.load(f))

# Normalize
a2_updates["requirement_title_clean"] = a2_updates["requirement_title"].str.lower().str.strip()
axis["section_title_clean"] = axis["section_title"].str.lower().str.strip()

# Match function
def is_matched(req_title, section_titles, threshold=0.85):
    return any(SequenceMatcher(None, req_title, sec).ratio() > threshold for sec in section_titles)

# Check for each requirement
axis_titles = axis["section_title_clean"].tolist()
a2_updates["exists_in_axis"] = a2_updates["requirement_title_clean"].apply(lambda x: is_matched(x, axis_titles))

# Filter out missing ones
missing_updates = a2_updates[~a2_updates["exists_in_axis"]].reset_index(drop=True)

# Save for disclosure generation
missing_updates.to_json("json/a2_axis_missing.json", orient="records", indent=2)
print(f"✅ Found {len(missing_updates)} missing updated disclosures")
print("📁 Saved to json/a2_axis_missing.json")


✅ Found 36 missing updated disclosures
📁 Saved to json/a2_axis_missing.json


In [2]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.docstore.document import Document
import os
import json

# Embeddings setup
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Convert JSONs into Document objects
def json_to_docs(json_data, source_tag):
    docs = []
    for item in json_data:
        content = "\n".join(f"{k}: {v}" for k, v in item.items())
        docs.append(Document(page_content=content, metadata={"source": source_tag}))
    return docs

# Load JSONs
datasets = {
    "a1": "json/basel_2019_extracted.json",
    "a2": "json/basel_2026_extracted.json",
    "axis": "json/axis_2025_extracted.json",
    "missing": "json/a2_axis_missing.json"
}

all_docs = []
for tag, file_path in datasets.items():
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        docs = json_to_docs(data, source_tag=tag)
        all_docs.extend(docs)

# Create ChromaDB and store
persist_directory = "vector_store"
vectordb = Chroma.from_documents(
    documents=all_docs,
    embedding=embedder,
    persist_directory=persist_directory
)
vectordb.persist()

print("✅ All data stored in ChromaDB at:", persist_directory)


C:\Users\krith\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ All data stored in ChromaDB at: vector_store


C:\Users\krith\AppData\Local\Temp\ipykernel_7664\3161246230.py:40: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [3]:
from langchain_chroma import Chroma

# Load 2026 updated regulations (already done)
vectordb_a2 = Chroma(persist_directory="chroma/a2_updated", embedding_function=embedder)

# Load Axis Bank 2025 report
vectordb_a3 = Chroma(persist_directory="chroma/a3_axis_report", embedding_function=embedder)


In [6]:
print(len(vectordb_a2.get()["documents"]))

0


In [5]:
missing_disclosures = []

for doc in vectordb_a2.get()["documents"]:
    results = vectordb_a3.similarity_search_with_score(doc, k=1)
    
    if not results:
        missing_disclosures.append(query_text)
    else:
        best_match, score = results[0]
        if score > 0.6:  # Tune this threshold. Lower = more inclusive, higher = stricter
            missing_disclosures.append(doc)

print(f"🧩 Missing Disclosures Found: {len(missing_disclosures)}")


🧩 Missing Disclosures Found: 0


In [22]:
%run backend/generated_axis_disclosures.py

  3%|██▎                                                                                | 1/36 [00:00<00:26,  1.33it/s]


[Item 0] Raw Output:
{
"requirement_title": "Disclosure of capital adequacy",
"category": "Capital",
"is_present_in_axis": false,
"importance_summary": "This requirement is important for banks and stakeholders as it provides transparency on an institution's ability to withstand financial shocks, ensuring its financial stability and confidence in the market.",
"generated_disclosure": {
"section_title": "Capital Adequacy Disclosure",
"narrative": "This section provides information on the institution's capital adequacy ratio (CAR), which represents the ratio of its total capital to its risk-weighted assets. The CAR is a key metric to assess an institution's ability to absorb potential losses and maintain its financial stability. The disclosure includes the institution's CAR, the composition of its capital, and any capital conservation buffer or countercyclical capital buffer.",
"suggested_metrics": [
"Capital Adequacy Ratio (CAR)",
"Total Capital",
"Risk-Weighted Assets",
"Capital Conser

Traceback (most recent call last):
  File "C:\Users\krith\AutomaticReportGenerator\backend\generated_axis_disclosures.py", line 75, in <module>
    parsed = json.loads(json_str)
             ^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
  6%|████▌                                                                              | 2/36 [00:01<00:24,  1.38it/s]


[Item 1] Raw Output:
Here is the response in the required JSON format:

{
"requirement_title": "Disclosure of Risk-Weighted Assets",
"category": "Risk-Weighted-Assets",
"is_present_in_axis": false,
"importance_summary": "This requirement is important as it enables stakeholders to understand the bank's risk profile and assess its capital adequacy. By disclosing risk-weighted assets, banks can provide transparency to investors, regulators, and customers, which is essential for maintaining trust and confidence in the financial system.",
"generated_disclosure": {
"section_title": "Risk-Weighted Assets Disclosure",
"narrative": "The institution discloses its risk-weighted assets, which represent the bank's risk exposure. Risk-weighted assets are calculated by multiplying the bank's assets by their corresponding risk weights. This disclosure provides insight into the bank's risk-taking activities, asset quality, and capital requirements.",
"suggested_metrics": [
"Risk-weighted assets as a p

  8%|██████▉                                                                            | 3/36 [00:02<00:22,  1.48it/s]

...


[Item 33] Raw Output:
{
"requirement_title": "Derivation of Shocks",
"category": "Risk-Weighted-Assets",
"is_present_in_axis": false,
"importance_summary": "This requirement is important for banks and stakeholders as it enables the derivation of shocks in interest rates, which are critical for calculating risk-weighted assets (RWA) under Basel III. Accurate calculation of RWA is essential for determining a bank's capital requirements, which in turn affects its ability to maintain financial stability and meet regulatory expectations.",
"generated_disclosure": {
"section_title": "Shocks in Interest Rates",
"narrative": "The derivation of shocks in interest rates is a critical component of risk-weighted asset calculations under Basel III. This requirement generates a time series of daily interest rates and calculates rate changes for a moving time window, providing a comprehensive view of interest rate movements. This information is essential for banks to accurately calculate their RWA a

 97%|███████████████████████████████████████████████████████████████████████████████▋  | 35/36 [01:26<00:04,  4.42s/it]


[Item 34] Raw Output:
{
"requirement_title": "Variable Caps for Interest Rate Shock Scenarios",
"category": "Risk-Weighted-Assets",
"is_present_in_axis": false,
"importance_summary": "This requirement is essential for banks as it ensures that they adequately capture potential losses arising from interest rate shocks, allowing for more accurate risk assessments and capital allocations.",
"generated_disclosure": {
"section_title": "Interest Rate Shock Scenario Caps",
"narrative": "The Basel III framework introduces variable caps for interest rate shock scenarios to provide a more comprehensive understanding of banks' interest rate risk. The caps set a floor of 100 bp and variable ceilings of 500 bp for short-term, 400 bp for parallel, and 300 bp for long-term interest rate shock scenarios, ensuring that banks adequately capture potential losses arising from these scenarios.",
"suggested_metrics": ["Interest Rate Shock Scenario Losses", "Capital Allocations for Interest Rate Risk", "Risk

Traceback (most recent call last):
  File "C:\Users\krith\AutomaticReportGenerator\backend\generated_axis_disclosures.py", line 75, in <module>
    parsed = json.loads(json_str)
             ^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [01:30<00:00,  2.51s/it]


[Item 35] Raw Output:
Here is the response in the requested JSON format:

{
"requirement_title": "Rounding of Interest Rate Shocks",
"category": "Credit Risk",
"is_present_in_axis": false,
"importance_summary": "This requirement is important for banks as it ensures transparency and consistency in the calculation of credit risk capital requirements. The rounding of interest rate shocks helps to simplify the process and reduce errors, ultimately contributing to a more accurate assessment of a bank's capital needs.",
"generated_disclosure": {
"section_title": "Rounding of Interest Rate Shocks",
"narrative": "As part of our credit risk management framework, we round the values from step 5 to the nearest multiple of 25 bp. This rounding is intended to simplify the calculation of credit risk capital requirements and reduce errors. By doing so, we ensure that our capital planning processes are transparent and consistent.",
"suggested_metrics": [
"Capital Requirements",
"Credit Risk Exposure"

In [12]:
with open("json/a2_axis_missing.json", "r", encoding="utf-8") as f:
    data = json.load(f)
print(len(data))

36
